# Automation for the analyst

A team of analysts prepares the monthly report on the prices of the product selected by the Board. Because they are aware you know Python, they asked you to automate the process. Talking to the team, you have set the following business conditions that enable process automation:

Three report parameters are available:
- **product_group_id**,
- **product**,
- **date**.

Assumptions for each parameter:

1. A parameter may have at most one value,
1. If the parameter is empty we return all records from the group,
1. We assume that the file is always prepared correctly (we want to practice report automation, not error handling).

Based on the above requirements:

1. load the  **config.xlsx** file using `openpyxl`,
1. prepare appropriate conditions to filter data from **product_cleaned.csv**,
1. based on the conditions filter the frame,
1. aggregate the data using a **pivot_table**:
   a) index-product, province,
   b) columns-dates,
   c) value-average product price,
   d) remember to remove 0,
6. save the file to the spreadsheet any way you want.

Hints:

1. You can save individual filtering conditions to variables and then use them all to filter `DataFrame`, the same as writing them all as before i.e. `df.loc[var1 & var2]`
1. If you decide to write with Pandas, be careful with the parameters passed to the function (what happens if you set `index=False`?). Link to the [documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_excel.html).

In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

df = pd.read_csv(
    r'C:\Users\UrosVukmanovic\CodersLab-Course-Python-Data-Analysis\Python Session_3\Session_3_-_exercise_files\01_Data\product_prices_cleaned_2.csv',
    sep=';',
    encoding='UTF-8',
    decimal='.'
)

In [2]:

wb = load_workbook(r'C:\Users\UrosVukmanovic\CodersLab-Course-Python-Data-Analysis\Python Session_3\Session_3_-_exercise_files\01_Data\config.xlsx', data_only=True)   # open existing file
print("Sheets:", wb.sheetnames)
ws = wb.active

Sheets: ['Sheet1', 'Sheet2']


In [3]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')


In [4]:
group_id = ws["B2"].value
product = ws["B3"].value
date = ws["B4"].value

print(group_id)
print(product)
print(date)
wb.close()

1
None
None


In [5]:
df_wo_duplicates = df.drop_duplicates()
df_wo_duplicates.head()

,province,product_types,currency,product_group_id,product_line,value,date,product,month,year,quarter
0,SUBCARPATHIA,NaN,PLN,2.0,pork ham cooked - per 1kg,21.37,2013-03-01,pork ham cooked - per 1kg,3.0,2013.0,1.0
1,ŁÓDŹ,NaN,PLN,4.0,bread - per 1kg,NaN,2018-02-01,bread - per 1kg,2.0,2018.0,1.0
2,KUYAVIA-POMERANIA,NaN,PLN,2.0,barley groats sausage - per 1kg,3.55,2019-12-01,barley groats sausage - per 1kg,12.0,2019.0,4.0
3,LOWER SILESIA,NaN,PLN,2.0,dressed chickens - per 1kg,6.14,2019-02-01,dressed chickens - per 1kg,2.0,2019.0,1.0
4,WARMIA-MASURIA,NaN,PLN,2.0,Italian head cheese - per 1kg,5.63,2002-03-01,Italian head cheese - per 1kg,3.0,2002.0,1.0


In [6]:
df_filtered= df_wo_duplicates.loc[
    (df['value'] > 0) &
    (df['value'] < 3000) &
    (df['date'] != '1888-0')
]

df_filtered.head()

,province,product_types,currency,product_group_id,product_line,value,date,product,month,year,quarter
0,SUBCARPATHIA,NaN,PLN,2.0,pork ham cooked - per 1kg,21.37,2013-03-01,pork ham cooked - per 1kg,3.0,2013.0,1.0
2,KUYAVIA-POMERANIA,NaN,PLN,2.0,barley groats sausage - per 1kg,3.55,2019-12-01,barley groats sausage - per 1kg,12.0,2019.0,4.0
3,LOWER SILESIA,NaN,PLN,2.0,dressed chickens - per 1kg,6.14,2019-02-01,dressed chickens - per 1kg,2.0,2019.0,1.0
4,WARMIA-MASURIA,NaN,PLN,2.0,Italian head cheese - per 1kg,5.63,2002-03-01,Italian head cheese - per 1kg,3.0,2002.0,1.0
5,HOLY CROSS,whole pickled cucumbers 0.9l - per 1pc.,PLN,1.0,NaN,0.28,2010-04-01,whole pickled cucumbers 0.9l - per 1pc.,4.0,2010.0,2.0


In [7]:
group_filter = df['product_group_id'] == (group_id if group_id else df['product_groupd_id'])
product_filter = df['product'] == (product if product else df['product'])
date_filter = df['date'] == (date if date else df['date'])
value_filter = df['value'] > 0


In [8]:
df_mod = df.loc[group_filter    &
                product_filter  &
                date_filter     &
                value_filter]

In [9]:
df_mod.head()


,province,product_types,currency,product_group_id,product_line,value,date,product,month,year,quarter
5,HOLY CROSS,whole pickled cucumbers 0.9l - per 1pc.,PLN,1.0,NaN,0.28,2010-04-01,whole pickled cucumbers 0.9l - per 1pc.,4.0,2010.0,2.0
12,POMERANIA,30% tomato concentrate - per 1kg,PLN,1.0,NaN,7.46,1999-10-01,30% tomato concentrate - per 1kg,10.0,1999.0,4.0
15,POLAND,whole pickled cucumbers 0.9l - per 1pc.,PLN,1.0,NaN,2.36,2004-12-01,whole pickled cucumbers 0.9l - per 1pc.,12.0,2004.0,4.0
26,LOWER SILESIA,frozen carrot and pea mix - per 1kg,EUR,1.0,NaN,2.78,2005-07-01,frozen carrot and pea mix - per 1kg,7.0,2005.0,3.0
37,MASOVIA,"apple juice, boxed - per 1l",PLN,1.0,NaN,1.91,2007-08-01,"apple juice, boxed - per 1l",8.0,2007.0,3.0


In [10]:
pt = pd.pivot_table(
    data=df_mod,
    index=['province', 'product'],
    columns=['date'],
    values=['value']
)
pt.head
pt.to_excel('03_Automation_for_analyst.xlsx')

Analytical Functions Session 3 summary

In [11]:
df = pd.read_csv(
    r'C:\Users\UrosVukmanovic\CodersLab-Course-Python-Data-Analysis\Python Session_3\Session_3_-_exercise_files\01_Data\product_prices_cleaned_2.csv',
    sep=';',
    encoding='UTF-8',
    decimal='.'
)

In [12]:
df = df.loc[
    (df['product'] == 'barley groats sausage - per 1kg') &
    (df['province'] == 'POLAND')
]
df.head()  # reviewing filtering correctness

,province,product_types,currency,product_group_id,product_line,value,date,product,month,year,quarter
888,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,4.34,2002-06-01,barley groats sausage - per 1kg,6.0,2002.0,2.0
3257,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,7.63,2012-03-01,barley groats sausage - per 1kg,3.0,2012.0,1.0
4384,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,7.24,2011-08-01,barley groats sausage - per 1kg,8.0,2011.0,3.0
4846,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,5.76,2008-08-01,barley groats sausage - per 1kg,8.0,2008.0,3.0
6033,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,3.70,2019-02-01,barley groats sausage - per 1kg,2.0,2019.0,1.0


Row Number Function

In [13]:
df_sorted = df.sort_values(by=['value'], ascending=False)
df_sorted['rn'] = df_sorted.groupby(by=['product']).cumcount()


In [14]:
df_sorted.head()


,province,product_types,currency,product_group_id,product_line,value,date,product,month,year,quarter,rn
58430,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.41,2019-01-01,barley groats sausage - per 1kg,1.0,2019.0,1.0,0
95873,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.40,2018-08-01,barley groats sausage - per 1kg,8.0,2018.0,3.0,1
27810,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.33,2018-10-01,barley groats sausage - per 1kg,10.0,2018.0,4.0,2
17846,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.24,2019-10-01,barley groats sausage - per 1kg,10.0,2019.0,4.0,3
44600,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.17,2018-06-01,barley groats sausage - per 1kg,6.0,2018.0,2.0,4


In [15]:
df_sorted.query("rn < 10")


,province,product_types,currency,product_group_id,product_line,value,date,product,month,year,quarter,rn
58430,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.41,2019-01-01,barley groats sausage - per 1kg,1.0,2019.0,1.0,0
95873,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.40,2018-08-01,barley groats sausage - per 1kg,8.0,2018.0,3.0,1
27810,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.33,2018-10-01,barley groats sausage - per 1kg,10.0,2018.0,4.0,2
17846,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.24,2019-10-01,barley groats sausage - per 1kg,10.0,2019.0,4.0,3
44600,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.17,2018-06-01,barley groats sausage - per 1kg,6.0,2018.0,2.0,4
26335,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.14,2019-09-01,barley groats sausage - per 1kg,9.0,2019.0,3.0,5
83380,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.12,2017-11-01,barley groats sausage - per 1kg,11.0,2017.0,4.0,6
46104,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.11,2018-02-01,barley groats sausage - per 1kg,2.0,2018.0,1.0,7
99469,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,8.98,2019-07-01,barley groats sausage - per 1kg,7.0,2019.0,3.0,8
79833,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,8.90,2018-12-01,barley groats sausage - per 1kg,12.0,2018.0,4.0,9


Cumulative Sum

In [16]:
df_sorted = df_sorted.sort_values(by=['value'], ascending=False)


In [17]:
df_sorted['cs'] = df_sorted.groupby(by=['product'])['value'].cumsum()
df_sorted.head()


,province,product_types,currency,product_group_id,product_line,value,date,product,month,year,quarter,rn,cs
58430,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.41,2019-01-01,barley groats sausage - per 1kg,1.0,2019.0,1.0,0,9.41
95873,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.40,2018-08-01,barley groats sausage - per 1kg,8.0,2018.0,3.0,1,18.81
27810,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.33,2018-10-01,barley groats sausage - per 1kg,10.0,2018.0,4.0,2,28.14
17846,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.24,2019-10-01,barley groats sausage - per 1kg,10.0,2019.0,4.0,3,37.38
44600,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,9.17,2018-06-01,barley groats sausage - per 1kg,6.0,2018.0,2.0,4,46.55


Cumulative Min


In [18]:
df_sorted = df_sorted.sort_values(by=['date'], ascending=True)


In [19]:
df_sorted['cummin'] = df_sorted.groupby(by=['product'])['value'].cummin()
df_sorted.head()


,province,product_types,currency,product_group_id,product_line,value,date,product,month,year,quarter,rn,cs,cummin
109100,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,3.40,1999-01-01,barley groats sausage - per 1kg,1.0,1999.0,1.0,247,1506.40,3.40
63584,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,3.59,1999-02-01,barley groats sausage - per 1kg,2.0,1999.0,1.0,237,1471.56,3.40
98728,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,3.25,1999-03-01,barley groats sausage - per 1kg,3.0,1999.0,1.0,249,1512.94,3.25
34091,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,3.41,1999-04-01,barley groats sausage - per 1kg,4.0,1999.0,2.0,245,1499.60,3.25
104708,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,3.50,1999-05-01,barley groats sausage - per 1kg,5.0,1999.0,2.0,243,1492.72,3.25


Cumulative Max

In [20]:
df_sorted = df_sorted.sort_values(by=['date'], ascending=True)
df_sorted['cummax'] = df_sorted.groupby(by=['product'])['value'].cummax()


df_sorted.head()


,province,product_types,currency,product_group_id,product_line,value,date,product,month,year,quarter,rn,cs,cummin,cummax
109100,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,3.40,1999-01-01,barley groats sausage - per 1kg,1.0,1999.0,1.0,247,1506.40,3.40,3.40
63584,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,3.59,1999-02-01,barley groats sausage - per 1kg,2.0,1999.0,1.0,237,1471.56,3.40,3.59
98728,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,3.25,1999-03-01,barley groats sausage - per 1kg,3.0,1999.0,1.0,249,1512.94,3.25,3.59
34091,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,3.41,1999-04-01,barley groats sausage - per 1kg,4.0,1999.0,2.0,245,1499.60,3.25,3.59
104708,POLAND,NaN,PLN,2.0,barley groats sausage - per 1kg,3.50,1999-05-01,barley groats sausage - per 1kg,5.0,1999.0,2.0,243,1492.72,3.25,3.59
